# Premium Prediction for Health insurance

# Step-1 : EDA

In [1]:
# Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# loading dataset
df = pd.read_excel("premium_rest_data.xlsx.xlsx")
df

FileNotFoundError: [Errno 2] No such file or directory: 'premium_rest_data.xlsx.xlsx'

# EDA + Data Cleaning

In [ ]:
df.info()

In [ ]:
df.Income_Level.unique()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.replace(" " , "_")

In [ ]:
df.columns

In [ ]:
df.columns = df.columns.str.lower()

In [ ]:
df.columns

## Handle the Null values

In [ ]:
df.isnull().sum()

In [ ]:
df.dropna(inplace = True)

In [ ]:
df.isnull().sum()

In [ ]:
df.shape

## Handle duplicate values

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe()

# Numerical columns : Univariate analysis

In [ ]:
## Issue-1 : number_of_dependants cannot be in negative

df[df.number_of_dependants < 0]

In [ ]:
df.number_of_dependants.unique()

In [ ]:
df['number_of_dependants'] = df.number_of_dependants.abs()

In [ ]:
df.number_of_dependants.unique()

In [ ]:
df.describe()

## Handle the Outliers

In [ ]:
numeric_columns = df.select_dtypes(include = ['int64', 'float64']).columns
numeric_columns                             

In [ ]:
for col in numeric_columns:
    sns.boxplot(x = df[col])
    plt.show()

## Handling the Age Outlier

In [ ]:
df[df.age > 100]['age'].count()

In [ ]:
df[df.age > 100]['age'].unique()

In [ ]:
median = df.age.median()
median

In [ ]:
df.loc[df.age > 100, 'age'] = median

In [ ]:
df[df.age > 100]['age'].count()

In [ ]:
df.describe()

In [ ]:
## Outliers in income_lakhs > 1CR (100)

df[df.income_lakhs > 100]['income_lakhs'].count()

In [ ]:
# Number of outliers using IQR

Q1, Q3 = df.income_lakhs.quantile([0.25, 0.75])
IQR = Q3 - Q1
LB = Q1 - 1.5 * IQR
UB = Q3 + 1.5 * IQR

LB, UB

In [ ]:
df[df.income_lakhs > UB]['income_lakhs'].count()

In [ ]:
df = df[df.income_lakhs <= 100]

In [ ]:
df.shape

In [ ]:
df.describe()

# EDA & DC on Caegorical data

In [ ]:
df.info()

In [ ]:
categorical_columns = df.select_dtypes(include = ['object']).columns
categorical_columns

In [ ]:
# checking sub-categories of each column
for col in categorical_columns:
    print(col, " : ", df[col].unique())

In [ ]:
df['smoking_status'] = df.smoking_status.replace({'Smoking=0' : 'No Smoking', 'Does Not Smoke' : 'No Smoking', 'Not Smoking' :  'No Smoking'})

In [ ]:
df.smoking_status.unique()

In [ ]:
df.shape

# Step-2 : Feature Engineering

In [ ]:
# checking sub-categories of each column
for col in categorical_columns:
    print(col, " : ", df[col].unique())

In [ ]:
risk_score = {
    'Diabetes' : 6,
    'Heart disease' : 8,
    'High blood pressure' : 6,
    'Thyroid' : 5,
    'No Disease' : 0
}

In [ ]:
df1 = df.copy()

In [ ]:
df1[['disease1', 'disease2']] = df1['medical_history'].str.split(" & ", expand = True)

In [ ]:
df1.disease1 = df1.disease1.map(risk_score)

In [ ]:
df1.disease2 = df1.disease2.map(risk_score)

In [ ]:
df1.disease1.isnull().sum()

In [ ]:
df1.disease2.isnull().sum()

In [ ]:
df1.disease2 = df1.disease2.fillna(0)

In [ ]:
df1

In [ ]:
df1['total_risk_score'] = df1['disease1'] + df1['disease2']

In [ ]:
df1 = df1.drop(['medical_history', 'disease1', 'disease2'], axis = 1)
df1

In [ ]:
# checking sub-categories of each column
categorical_columns = df1.select_dtypes(include = ['object']).columns
categorical_columns

for col in categorical_columns:
    print(col, " : ", df1[col].unique())

### Apply Ordinal encoding to convert insurance_plan & income_level into neumerical

In [ ]:
# use ordinal encoding when Order/Ranking matters

In [ ]:
from sklearn.preprocessing import OrdinalEncoder

In [ ]:
encoder = OrdinalEncoder(categories = [['Bronze','Silver','Gold']])
df1['insurance_plan'] = encoder.fit_transform(df1[['insurance_plan']])

In [ ]:
encoder = OrdinalEncoder(categories = [['<10L','10L - 25L','25L - 40L', '> 40L']])
df1['income_level'] = encoder.fit_transform(df1[['income_level']])

In [ ]:
df1

## Apply Label encoder on region and bmi_category

In [ ]:
# Order/Ranking don't matter in Label encoding

In [ ]:
from sklearn.preprocessing import LabelEncoder

In [ ]:
encoder = LabelEncoder()
for col in ['region', 'bmi_category']:
    df1[col] = encoder.fit_transform(df1[col])

In [ ]:
df1

### convert into numeric using one-hot encoding

In [ ]:
df.smoking_status.unique()

In [ ]:
df1 = pd.get_dummies(df1, columns = ['gender', 'marital_status', 'smoking_status', 'employment_status'], drop_first = True, dtype = int)

In [ ]:
df1

## finding correlation 

In [ ]:
cm = df1.corr()

plt.figure(figsize = (20,12))
sns.heatmap(cm, annot = True)



# income_level = income_lakhs   & number_of_dependents = marital_status_unmarried

In [ ]:
# income_level = income_lakhs   & number_of_dependents = marital_status_unmarried

## feature scaling

In [ ]:
x = df1.drop(["income_level", "marital_status_Unmarried", "annual_premium_amount"], axis = 1)
y = df1.annual_premium_amount
x.columns

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size = 0.2, random_state = 42)

In [ ]:
from sklearn.preprocessing import MinMaxScaler   # 0 to 1

scaler = MinMaxScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

In [ ]:
x_train_scaled.shape

In [ ]:
x_test_scaled.shape

# Step-3 : Apply ML models

### Linear Regression

In [ ]:
from sklearn.linear_model import LinearRegression

linear_obj = LinearRegression()

linear_obj.fit(x_train_scaled, y_train)

In [ ]:
lr_score = linear_obj.score(x_test_scaled, y_test)
print(lr_score)

### model-2 : XGBoost

In [ ]:
from xgboost import XGBRegressor

In [ ]:
XGB_obj = XGBRegressor()

In [ ]:
from sklearn.model_selection import RandomizedSearchCV

param_grid = {
                'n_estimators' : [50, 60, 70, 80, 90, 100],
                'max_depth' : [3,4,5,6,7,8,9,10]
}

random_search = RandomizedSearchCV(XGB_obj, param_grid, cv = 5)

In [ ]:
random_search.fit(x_train_scaled, y_train)

In [ ]:
random_search.best_score_

# CONGRATULATIONS - TARGET-1 ACHIEVED

# TARGET-2

In [ ]:
y_pred = random_search.predict(x_test_scaled)
y_pred

In [ ]:
residuals = y_pred - y_test

residuals_pct = (residuals / y_test) * 100
residuals_pct

In [ ]:
result_df = pd.DataFrame({
    'Actual' : y_test,
    'Predicted' : y_pred,
    'diff' : residuals,
    'diff_pct' : residuals_pct })

result_df 

In [ ]:
sns.histplot(result_df ['diff_pct'], kde = True)

In [ ]:
extreme_result_df = result_df[np.abs(result_df.diff_pct) > 10]
extreme_result_df 

### Create extreme_result_df with all the features

In [ ]:
extreme_error_df = x_test.loc[extreme_result_df.index]
extreme_error_df

error_percentage = (extreme_error_df.shape[0] / x_test.shape[0]) * 100
print(error_percentage)

In [ ]:
extreme_error_df.age.unique()

### Finding the problem at Data level

In [ ]:
df1.columns

In [ ]:
# We need to find due to which feature "predicted - actual" difference is high

In [ ]:
sns.histplot(x_test['age'], kde = True)
sns.histplot(extreme_error_df['age'], color = 'blue', kde = True)

In [ ]:
sns.histplot(x_test['region'], kde = True)
sns.histplot(extreme_error_df['region'], color = 'blue', kde = True)

In [ ]:
x_test.shape

In [ ]:
extreme_error_df.shape

In [ ]:
# if the shape of error data and actual data is different for a feature
# that's means problem is occuring due to that particular feature

In [ ]:
sns.histplot(x_test['number_of_dependants'], kde = True)
sns.histplot(extreme_error_df['number_of_dependants'], color = 'blue', kde = True)

In [ ]:
type(x_test)

In [ ]:
x_test[x_test.age < 24]

In [ ]:
# People with age less then 24 are unpredicatable

# people whose age < 24, their the "predict - actual' difference is more

# Data Segmentation